# 03 — Build Dataset Arrays

Stacks all per-subject feature files into `X.npy` and aligned target arrays `y_regression.npy` + `y_network.npy`.

**X** shape: `(N, 2, 48, 48, 32)` — N subjects, 2 channels (mean BOLD + fALFF)

**y_regression** shape: `(N, 3)` — trait anxiety, chronic stress, neuroticism (z-scored)

**y_network** shape: `(N, 3)` — DMN, Salience, ECN activation levels (0–1, derived from ROI signal)

Also saves `wb_stats.json` (mean/std per regression target) so the FastAPI backend can convert z-scores back to real units.

In [ ]:
!pip install -q nibabel nilearn numpy pandas scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd

BASE_DIR = '/content/drive/MyDrive/lemon'
FEAT_DIR = f'{BASE_DIR}/features'

pheno = pd.read_csv(f'{BASE_DIR}/phenotypic.csv')
with open(f'{BASE_DIR}/target_columns.json') as f:
    col_map = json.load(f)

feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('.npy')])
print(f'Feature files: {len(feat_files)}')
print(f'Target columns: {col_map}')

In [ ]:
# ── ROI coordinates for resting-state network proxies ─────────────────────────
# MNI coordinates (mm) for seed regions of each network.
# We extract the mean signal within a 6mm sphere around each seed
# and use it as a proxy for network activation level.
#
# DMN seeds:  medial prefrontal cortex + posterior cingulate cortex
# Salience:   anterior insula + anterior cingulate cortex
# ECN:        dorsolateral prefrontal cortex (L) + posterior parietal cortex (L)

NETWORK_SEEDS = {
    'DMN':      [(-1, 52, -6),    (-1, -53, 26)],   # mPFC, PCC
    'Salience': [(38, 6, -4),     (4, 28, 26)],      # R insula, dACC
    'ECN':      [(-44, 28, 28),   (-30, -56, 46)],   # L dlPFC, L PPC
}

TARGET_AFFINE = np.diag([4.0, 4.0, 4.5, 1.0])
TARGET_SHAPE  = (48, 48, 32)

def mni_to_vox(mni_coord, affine):
    """Convert MNI mm coordinate to voxel index."""
    inv = np.linalg.inv(affine)
    vox = (inv @ np.array([*mni_coord, 1]))[:3]
    return np.round(vox).astype(int)

def sphere_mask(center_vox, radius_vox, shape):
    """Boolean mask for a sphere of given radius around center_vox."""
    x, y, z = np.mgrid[:shape[0], :shape[1], :shape[2]]
    dist = np.sqrt((x-center_vox[0])**2 + (y-center_vox[1])**2 + (z-center_vox[2])**2)
    return dist <= radius_vox

# Pre-compute seed masks in voxel space
seed_masks = {}
for net, seeds in NETWORK_SEEDS.items():
    masks = []
    for coord in seeds:
        vox = mni_to_vox(coord, TARGET_AFFINE)
        vox = np.clip(vox, 0, np.array(TARGET_SHAPE)-1)
        masks.append(sphere_mask(vox, radius_vox=1.5, shape=TARGET_SHAPE))  # ~6mm sphere
    # Union of seed masks for this network
    seed_masks[net] = np.any(masks, axis=0)
    print(f'{net}: {seed_masks[net].sum()} voxels in mask')

print('Seed masks ready ✓')

In [ ]:
from tqdm import tqdm

X_list        = []
y_reg_list    = []
y_net_list    = []
subject_ids   = []
skipped       = []

# Index phenotypic data by participant_id for fast lookup
pheno_idx = pheno.set_index('participant_id')

for fname in tqdm(feat_files, desc='Building dataset'):
    sub = fname.replace('.npy', '')  # e.g. 'sub-010002'
    pid = sub if sub.startswith('sub-') else f'sub-{sub}'

    # ── Look up phenotypic targets ────────────────────────────────────────────
    if pid not in pheno_idx.index:
        skipped.append(sub)
        continue
    row = pheno_idx.loc[pid]
    targets = [row.get(col_map.get(k), np.nan)
               for k in ['trait_anxiety', 'chronic_stress', 'neuroticism']]
    if any(np.isnan(t) for t in targets):
        skipped.append(sub)
        continue

    # ── Load feature array ────────────────────────────────────────────────────
    feats = np.load(f'{FEAT_DIR}/{fname}')  # (2, 48, 48, 32)

    # ── Derive network activation from fALFF channel (channel 1) ─────────────
    falff = feats[1]  # (48, 48, 32)
    net_vals = []
    for net in ['DMN', 'Salience', 'ECN']:
        mask = seed_masks[net]
        net_vals.append(float(falff[mask].mean()))

    X_list.append(feats)
    y_reg_list.append(targets)
    y_net_list.append(net_vals)
    subject_ids.append(sub)

print(f'Included: {len(X_list)}  Skipped: {len(skipped)}')
if skipped:
    print('Skipped:', skipped[:10])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

X        = np.array(X_list, dtype=np.float32)      # (N, 2, 48, 48, 32)
y_reg_raw = np.array(y_reg_list, dtype=np.float32) # (N, 3) — raw scores
y_net_raw = np.array(y_net_list, dtype=np.float32) # (N, 3) — raw fALFF means

# ── Z-score regression targets & save stats for inference ────────────────────
wb_mean = y_reg_raw.mean(axis=0)
wb_std  = y_reg_raw.std(axis=0)
y_reg   = (y_reg_raw - wb_mean) / (wb_std + 1e-8)

# Save stats so the FastAPI backend can invert z-scores at inference time
wb_stats = {
    'targets': ['trait_anxiety', 'chronic_stress', 'neuroticism'],
    'mean':    wb_mean.tolist(),
    'std':     wb_std.tolist(),
}
with open(f'{BASE_DIR}/wb_stats.json', 'w') as f:
    json.dump(wb_stats, f, indent=2)

# ── Scale network activations to 0–1 ─────────────────────────────────────────
scaler = MinMaxScaler()
y_net  = scaler.fit_transform(y_net_raw).astype(np.float32)

# Save scaler params
import pickle
with open(f'{BASE_DIR}/net_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f'X shape:     {X.shape}')
print(f'y_reg shape: {y_reg.shape}  — z-scored [trait_anxiety, chronic_stress, neuroticism]')
print(f'y_net shape: {y_net.shape}  — 0–1 scaled [DMN, Salience, ECN]')
print(f'\nWellbeing stats:')
for i, name in enumerate(['trait_anxiety', 'chronic_stress', 'neuroticism']):
    print(f'  {name}: mean={wb_mean[i]:.2f}, std={wb_std[i]:.2f}')

In [ ]:
# ── Save arrays ───────────────────────────────────────────────────────────────
np.save(f'{BASE_DIR}/X.npy',         X)
np.save(f'{BASE_DIR}/y_regression.npy', y_reg)
np.save(f'{BASE_DIR}/y_network.npy',    y_net)

with open(f'{BASE_DIR}/subject_ids_final.json', 'w') as f:
    json.dump(subject_ids, f)

size_mb = (X.nbytes + y_reg.nbytes + y_net.nbytes) / 1e6
print(f'Total dataset size: {size_mb:.0f} MB')
print('\nFiles saved:')
print('  X.npy, y_regression.npy, y_network.npy')
print('  wb_stats.json, net_scaler.pkl, subject_ids_final.json')
print('\nNext step: open 04_train.ipynb')